# Lab 05 — Lakeflow Declarative Pipelines
## 00 — Environment Setup

This notebook performs the **one-time infrastructure setup** for Lab 05.

### Responsibilities
- Validate the target Unity Catalog catalog
- Create the Lab 05 schema if required
- Create the Lab 05 Unity Catalog volume
- Create source-data directories used by the lab
- Validate that the required directories exist

### This notebook intentionally does **not**
- Create Bronze, Silver, or Gold pipeline datasets
- Run the Lakeflow declarative pipeline
- Create or manage streaming checkpoints
- Create or manage Auto Loader schema locations
- Download Citi Bike source data

Those responsibilities are kept separate so that the project follows a clean separation of concerns:

**Setup → Source preparation → Declarative pipeline → Validation**

The Lakeflow pipeline will own its Bronze, Silver, and Gold datasets.


## 1. Runtime parameters

The manual setup notebook uses five infrastructure parameters:

- `catalog`
- `schema`
- `volume_name` — managed reference/test-data volume
- `streaming_volume_name` — external volume used by Auto Loader
- `external_location_name` — parent Unity Catalog external location

The extra two parameters are required because the streaming Auto Loader source is intentionally separated from managed reference storage.


In [0]:
dbutils.widgets.text("catalog", "dbr_dev", "Catalog")
dbutils.widgets.text("schema", "parvinbadalov", "Schema")
dbutils.widgets.text("volume_name", "lab05_lakeflow", "Managed volume")
dbutils.widgets.text(
    "streaming_volume_name",
    "lab05_lakeflow_streaming",
    "Streaming external volume"
)
dbutils.widgets.text(
    "external_location_name",
    "parvinbadalov_root_location",
    "External location"
)

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
volume_name = dbutils.widgets.get("volume_name").strip()
streaming_volume_name = dbutils.widgets.get(
    "streaming_volume_name"
).strip()
external_location_name = dbutils.widgets.get(
    "external_location_name"
).strip()

print(f"Catalog                  : {catalog}")
print(f"Schema                   : {schema}")
print(f"Managed volume           : {volume_name}")
print(f"Streaming external volume: {streaming_volume_name}")
print(f"External location        : {external_location_name}")


Catalog                  : dbr_dev
Schema                   : parvinbadalov
Managed volume           : lab05_lakeflow
Streaming external volume: lab05_lakeflow_streaming
External location        : parvinbadalov_root_location


## 2. Validate identifiers and define storage paths

Before using widget values in SQL object names, validate that they contain only safe identifier characters.

The Lab 05 volume will contain only source/test folders:

```text
/Volumes/<catalog>/<schema>/<volume_name>/
├── landing/
│   └── station_status/
├── reference/
└── test_data/
```

There is deliberately **no manually managed `checkpoints/` directory** in this design.

The goal of Lab 05 is to use Lakeflow declarative pipeline behavior rather than recreate the manual checkpoint-management pattern used in classic Structured Streaming.


In [0]:
import re

IDENTIFIER_PATTERN = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def validate_identifier(name: str, label: str) -> None:
    if not IDENTIFIER_PATTERN.fullmatch(name):
        raise ValueError(
            f"Invalid {label}: {name!r}. "
            "Use letters, numbers, and underscores only."
        )


for value, label in [
    (catalog, "catalog"),
    (schema, "schema"),
    (volume_name, "managed volume name"),
    (streaming_volume_name, "streaming volume name"),
    (external_location_name, "external location name"),
]:
    validate_identifier(value, label)


volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"
streaming_volume_path = (
    f"/Volumes/{catalog}/{schema}/{streaming_volume_name}"
)

status_landing_path = (
    f"{streaming_volume_path}/landing/station_status"
)
reference_path = f"{volume_path}/reference"
test_data_path = f"{volume_path}/test_data"

print(f"Managed volume root : {volume_path}")
print(f"Streaming volume root: {streaming_volume_path}")
print(f"Streaming landing    : {status_landing_path}")
print(f"Reference data       : {reference_path}")
print(f"Test data            : {test_data_path}")


Managed volume root : /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow
Streaming volume root: /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming
Streaming landing    : /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status
Reference data       : /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow/reference
Test data            : /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow/test_data


## 3. Validate the catalog and create Lab 05 structural objects

The catalog is treated as shared infrastructure, so this notebook **does not create it**.

Instead:
1. Verify that the catalog exists and is accessible.
2. Create the personal Lab 05 schema if it does not already exist.
3. Create the Lab 05 managed volume if it does not already exist.

Using `IF NOT EXISTS` makes this setup step safe to rerun.


In [0]:
available_catalogs = {
    row.catalog
    for row in spark.sql("SHOW CATALOGS").collect()
}

if catalog not in available_catalogs:
    raise ValueError(
        f"Catalog '{catalog}' does not exist or is not accessible."
    )

print(f"✅ Catalog exists: {catalog}")

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`"
)

print(f"✅ Schema ready: {catalog}.{schema}")

# Managed volume: reference data + test fixtures.
spark.sql(
    f'''
    CREATE VOLUME IF NOT EXISTS
    `{catalog}`.`{schema}`.`{volume_name}`
    '''
)

print(f"✅ Managed volume ready: {catalog}.{schema}.{volume_name}")

# Resolve the exact parent external-location URL instead of hardcoding it.
location_rows = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{external_location_name}`"
).collect()

if not location_rows:
    raise ValueError(
        f"External location not found: {external_location_name}"
    )

external_location_url = str(location_rows[0]["url"]).rstrip("/")
streaming_storage_location = (
    f"{external_location_url}/{streaming_volume_name}"
)

spark.sql(
    f'''
    CREATE EXTERNAL VOLUME IF NOT EXISTS
    `{catalog}`.`{schema}`.`{streaming_volume_name}`
    LOCATION '{streaming_storage_location}'
    '''
)

print(
    f"✅ External streaming volume ready: "
    f"{catalog}.{schema}.{streaming_volume_name}"
)
print(f"   Storage: {streaming_storage_location}")


✅ Catalog exists: dbr_dev
✅ Schema ready: dbr_dev.parvinbadalov
✅ Managed volume ready: dbr_dev.parvinbadalov.lab05_lakeflow
✅ External streaming volume ready: dbr_dev.parvinbadalov.lab05_lakeflow_streaming
   Storage: abfss://parvinbadalov@dlspl21databricks.dfs.core.windows.net/lab05_lakeflow_streaming


## 4. Create source and test-data directories

Only directories required for external/source files are created here.

### `landing/station_status/`
Receives timestamped Citi Bike `station_status` JSON snapshots. Each producer execution will later write **one immutable snapshot file**.

### `reference/`
Stores the batch/reference `station_information.json` file.

### `test_data/`
Stores controlled invalid or test fixtures used to prove expectation behavior.

Bronze, Silver, and Gold tables are **not** created here. They will be declared inside `pipeline/bronze.py`, `pipeline/silver.py`, and `pipeline/gold.py`.


In [0]:
directories = [
    status_landing_path,
    reference_path,
    test_data_path,
]

for path in directories:
    dbutils.fs.mkdirs(path)
    print(f"✅ Ready: {path}")


✅ Ready: /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status
✅ Ready: /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow/reference
✅ Ready: /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow/test_data


## 5. Inspect the created directory structure

This step provides quick visual evidence that the infrastructure is ready before source preparation begins.


In [0]:
display(dbutils.fs.ls(volume_path))


path,name,size,modificationTime
dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow/landing/,landing/,0,1786921988394
dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow/reference/,reference/,0,1786921988394
dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow/test_data/,test_data/,0,1786921988394


In [0]:
display(dbutils.fs.ls(f"{volume_path}/landing"))


path,name,size,modificationTime
dbfs:/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow/landing/station_status/,station_status/,0,1786921995943


## 6. Final setup validation

Instead of assuming that directory creation succeeded, perform explicit validation.

This follows the same principle used in previous labs: setup should finish with a clear **PASS/FAIL** result rather than leaving infrastructure problems to surface later inside the production pipeline.


In [0]:
required_directories = {
    "station_status_external_landing": status_landing_path,
    "reference_managed": reference_path,
    "test_data_managed": test_data_path,
}

validation_results = []

for name, path in required_directories.items():
    try:
        dbutils.fs.ls(path)
        validation_results.append((name, path, "PASS"))
    except Exception as exc:
        validation_results.append(
            (name, path, f"FAIL: {exc}")
        )

validation_df = spark.createDataFrame(
    validation_results,
    ["check", "path", "result"]
)

display(validation_df)


check,path,result
station_status_external_landing,/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status,PASS
reference_managed,/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow/reference,PASS
test_data_managed,/Volumes/dbr_dev/parvinbadalov/lab05_lakeflow/test_data,PASS


In [0]:
failed_checks = [
    row
    for row in validation_results
    if not row[2].startswith("PASS")
]

if failed_checks:
    raise AssertionError(
        f"Lab 05 setup failed: {failed_checks}"
    )

print("✅ LAB 05 SETUP PASSED")
print()
print(f"Managed volume     : {volume_path}")
print(f"Streaming volume   : {streaming_volume_path}")
print(f"Streaming landing  : {status_landing_path}")
print(f"Reference data     : {reference_path}")
print(f"Test data          : {test_data_path}")


✅ LAB 05 SETUP PASSED

Managed volume     : /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow
Streaming volume   : /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming
Streaming landing  : /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow_streaming/landing/station_status
Reference data     : /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow/reference
Test data          : /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow/test_data


## Expected result

A successful run should end with:

```text
✅ LAB 05 SETUP PASSED

Volume: /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow
Streaming landing: /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow/landing/station_status
Reference data: /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow/reference
Test data: /Volumes/dbr_dev/parvinbadalov/lab05_lakeflow/test_data
```

### Ownership boundary after this notebook

| Component | Owner |
|---|---|
| Catalog validation | `lab05_00_setup` |
| Schema | `lab05_00_setup` |
| Volume | `lab05_00_setup` |
| Landing/reference/test directories | `lab05_00_setup` |
| Citi Bike source download | `lab05_01_source_preparation` |
| Streaming snapshot producer | `tools/citibike_status_producer.py` |
| Bronze datasets | Lakeflow pipeline |
| Silver datasets + expectations | Lakeflow pipeline |
| Gold dataset | Lakeflow pipeline |
| Streaming state/checkpoints | Lakeflow |
| Final validation | `lab05_02_validation` |

### Next step

Continue with `notebooks/lab05_01_source_preparation.ipynb`. That notebook will download `station_information.json`, fetch initial `station_status` snapshots, and profile both sources before we design schemas and expectations.
